# 03 - Model Experiments

This notebook trains and evaluates the two main modeling approaches:
1. Fine-tuned ResNet-18 (reference implementation)
2. DenseNet-Attention (student-developed model)

Both are compared on the same test set with identical preprocessing.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import get_dataloaders
from src.models import get_model, get_optimizer
from src.train import run_experiment
from src.evaluate import (
    evaluate_model, plot_confusion_matrix, plot_roc_curve,
    plot_training_history, plot_metrics_comparison,
    generate_gradcam, plot_gradcam_grid,
)
from src.utils import set_seed, get_device, load_config, save_results, count_parameters

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]

print(f"Device: {device}")

In [ ]:
set_seed(SEED)

dataloaders = get_dataloaders(
    DATA_ROOT,
    augmentation="standard",
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    val_split=config["data"]["val_split"],
    num_workers=0,
    seed=SEED,
)

print(f"Dataset info: {dataloaders['info']}")

## Model 1: Fine-tuned ResNet-18

In [ ]:
set_seed(SEED)

resnet_model = get_model(
    "resnet18_finetune",
    pretrained=True,
    dropout=config["model"]["dropout"],
)

params = count_parameters(resnet_model)
print(f"ResNet-18 parameters: {params['total']:,} total, {params['trainable']:,} trainable")

resnet_config = config.copy()
resnet_config["model"]["use_focal_loss"] = False

resnet_history = run_experiment(
    model_name="resnet18_finetune",
    model=resnet_model,
    dataloaders=dataloaders,
    device=device,
    config=resnet_config,
    experiment_name="resnet18_finetune",
)

In [ ]:
fig = plot_training_history(resnet_history["history"], title="ResNet-18 Fine-tune")
plt.savefig("../results/resnet18_training.png", bbox_inches="tight")
plt.show()

In [ ]:
resnet_test = evaluate_model(resnet_model, dataloaders["test"], device)

print("ResNet-18 Fine-tune - Test Results:")
for key in ["auroc", "f1_macro", "sensitivity", "specificity", "precision", "npv", "accuracy"]:
    print(f"  {key:>15s}: {resnet_test['metrics'][key]:.4f}")

print(f"\nThreshold analysis:")
for key, val in resnet_test["threshold_analysis"].items():
    print(f"  {key}: {val:.4f}")

## Model 2: DenseNet-Attention (Student-Developed)

In [ ]:
set_seed(SEED)

densenet_model = get_model(
    "densenet_attention",
    pretrained=True,
    dropout=config["model"]["dropout"],
    use_attention=config["model"]["use_attention"],
)

params = count_parameters(densenet_model)
print(f"DenseNet-Attention parameters: {params['total']:,} total, {params['trainable']:,} trainable")

densenet_history = run_experiment(
    model_name="densenet_attention",
    model=densenet_model,
    dataloaders=dataloaders,
    device=device,
    config=config,
    experiment_name="densenet_attention",
)

In [ ]:
fig = plot_training_history(densenet_history["history"], title="DenseNet-Attention")
plt.savefig("../results/densenet_attention_training.png", bbox_inches="tight")
plt.show()

In [ ]:
densenet_test = evaluate_model(densenet_model, dataloaders["test"], device)

print("DenseNet-Attention - Test Results:")
for key in ["auroc", "f1_macro", "sensitivity", "specificity", "precision", "npv", "accuracy"]:
    print(f"  {key:>15s}: {densenet_test['metrics'][key]:.4f}")

print(f"\nThreshold analysis:")
for key, val in densenet_test["threshold_analysis"].items():
    print(f"  {key}: {val:.4f}")

## Confusion matrices side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_confusion_matrix(
    resnet_test["y_true"], resnet_test["y_proba"],
    title="ResNet-18", ax=axes[0],
)
plot_confusion_matrix(
    densenet_test["y_true"], densenet_test["y_proba"],
    title="DenseNet-Attention", ax=axes[1],
)

plt.tight_layout()
plt.savefig("../results/model_confusion_matrices.png", bbox_inches="tight")
plt.show()

## ROC Curve Comparison

In [ ]:
all_results = {
    "ResNet-18": resnet_test,
    "DenseNet-Attention": densenet_test,
}

fig, ax = plt.subplots(figsize=(7, 6))
plot_roc_curve(all_results, ax=ax)
plt.savefig("../results/model_roc_curves.png", bbox_inches="tight")
plt.show()

## Grad-CAM Visualization

We visualize the regions of the X-ray that each model attends to,
to verify the clinical plausibility of the learned features.

In [ ]:
test_iter = iter(dataloaders["test"])
sample_images, sample_labels = next(test_iter)
sample_images = sample_images[:8]
sample_labels = sample_labels[:8]

# Grad-CAM for DenseNet-Attention
# Target the last convolutional layer of the DenseNet backbone
target_layer = densenet_model.backbone[-1]

heatmaps = generate_gradcam(densenet_model, sample_images, target_layer, device)

with torch.no_grad():
    preds = torch.sigmoid(densenet_model(sample_images.to(device)).squeeze()).cpu().numpy()

fig = plot_gradcam_grid(sample_images, heatmaps, sample_labels.numpy(), preds, n_samples=8)
plt.suptitle("DenseNet-Attention: Grad-CAM Visualizations", fontsize=14, y=1.02)
plt.savefig("../results/gradcam_densenet.png", bbox_inches="tight")
plt.show()

In [ ]:
# Grad-CAM for ResNet-18
target_layer_resnet = resnet_model.resnet.layer4[-1]

heatmaps_resnet = generate_gradcam(resnet_model, sample_images, target_layer_resnet, device)

with torch.no_grad():
    preds_resnet = torch.sigmoid(resnet_model(sample_images.to(device)).squeeze()).cpu().numpy()

fig = plot_gradcam_grid(sample_images, heatmaps_resnet, sample_labels.numpy(), preds_resnet, n_samples=8)
plt.suptitle("ResNet-18: Grad-CAM Visualizations", fontsize=14, y=1.02)
plt.savefig("../results/gradcam_resnet.png", bbox_inches="tight")
plt.show()

## Save all results

In [ ]:
save_results(
    {
        "resnet18": resnet_test["metrics"],
        "densenet_attention": densenet_test["metrics"],
        "resnet18_threshold": resnet_test["threshold_analysis"],
        "densenet_attention_threshold": densenet_test["threshold_analysis"],
    },
    "model_results",
    output_dir="../results",
)
print("Model results saved.")